# 第14周 · 练习：数参数、从零写 LoRA、验证 KL

五道题，前三道围绕"可训练参数"，后两道是数学工作坊的公式验证：

| 题 | 内容 | 自查 |
| --- | --- | --- |
| E1 | 数可训练参数：写 `count_trainable`，再手算一个 Transformer 层有多少参数 | `ck.count_params`、`ck.trainable_params` |
| E2 | 从零写 `LoRALinear`，并数 LoRA 的参数 | `ck.lora_init_identity`、`ck.lora_param_count` |
| E3 | 把你的 `LoRALinear` 装进 DistilBERT，真的训练一次 | `ck.scratch_lora_accuracy` |
| E4 | 两个一维高斯的解析 KL | `ck.gaussian_kl` |
| E5 | 用蒙特卡洛估计 KL，并比较两个方向的估计噪声 | `ck.mc_kl` |

E3 在 CPU 上要训练约一分钟；在 Colab 上记得先打开 GPU（代码执行程序 → 更改运行时类型 → T4 GPU）。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 第 14 周追加：Hugging Face transformers（5.x 写法）与 PEFT。Colab 预装了 transformers，一般只需装 peft
for pkg, spec in (("transformers", "transformers>=5.22,<6"), ("peft", "peft>=0.21,<1")):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec], check=True)
print("transformers / peft 已就绪")

In [ ]:
import os, time, math
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from zero2researcher import data, experiments
from zero2researcher.checks import week14 as ck
from zero2researcher.checks import report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   # 本周跳过 MPS（见讲义）
transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

## E1 · 数可训练参数

**E1a**：写 `count_trainable(model)`：返回模型里**可训练**参数（`requires_grad` 为 True）的元素总数。

提示：`model.parameters()` 逐个给出参数张量；一个张量有多少个元素用 `p.numel()`。

In [ ]:
def count_trainable(model):
    # 在这里写你的代码
    raise NotImplementedError


ck.count_params(count_trainable)

**E1b**：有一种常见的折中配置叫"解冻最后一层 + 分类头"：冻结 embeddings 和底部 5 个 Transformer 层，只训练第 6 层和分类头。

1. **先手算**：一个 DistilBERT Transformer 层里有 4 个 768×768 的注意力线性层（`q_lin / k_lin / v_lin / out_lin`，都带 bias）、一个 768→3072 和一个 3072→768 的前馈层（带 bias）、两个 LayerNorm（每个有 768 个 weight 和 768 个 bias）。分类头 `pre_classifier`（768→768）+ `classifier`（768→2）。把 `layer_params`、`head_params` 写成算式；
2. 再写冻结代码，用 `count_trainable` 数出实际值 `n`，看和手算是否一致。

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
REVISION = "12040accade4e8a0f71eabdb258fecc2e7e948be"   # 固定模型版本（Hub 上的提交号）：模型仓库以后更新，课程里的数字也能复现
# 在这里写你的代码
raise NotImplementedError
predicted = layer_params + head_params
print(f"手算：每层 {layer_params:,} | 分类头 {head_params:,} | 最后一层 + 头 = {predicted:,}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2)
# 在这里写你的代码
raise NotImplementedError
n = count_trainable(model)
print(f"实际可训练 {n:,}（全模型 {sum(p.numel() for p in model.parameters()):,}）")
ck.trainable_params(n, model)
report("手算与实际一致", predicted == n, f"手算 {predicted:,}，实际 {n:,}",
       "逐项核对：注意力 4 个线性层、前馈 2 个线性层、2 个 LayerNorm，每个线性层都别漏了 bias")

**追问**（一句话）：一个 Transformer 层里，注意力和前馈网络谁的参数更多？LoRA 论文只改 `q_lin` 和 `v_lin`，它们占一层参数的几分之几？

（在这里回答）

## E2 · 从零写 LoRALinear

按讲义 S2 的公式实现（Hu et al. 2021 式 (3) $h = W_0x + BAx$，加上紧随其后的正文 §4.1 里的 $\alpha/r$ 缩放）：

$$h = W_0 x + \frac{\alpha}{r} B A x, \qquad A \in \mathbb{R}^{r \times d_\text{in}}\ \text{（高斯初始化）},\quad B \in \mathbb{R}^{d_\text{out} \times r}\ \text{（零初始化）}$$

要求（自查会逐条检查）：

- 构造写法 `LoRALinear(base, r, alpha)`，`base` 是一个 `nn.Linear`；`base` 的参数全部冻结；
- 两个低秩矩阵存为 `self.A`、`self.B`（`nn.Parameter`），形状如上；
- 前向对任意形状 `(..., d_in)` 的输入都成立——DistilBERT 里传进来的是 `(batch, 序列长度, 768)`。

**先别看讲义**，自己写一遍。

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__()
        # 在这里写你的代码
        raise NotImplementedError

    def forward(self, x):
        # 在这里写你的代码
        raise NotImplementedError


ck.lora_init_identity(LoRALinear)

**E2b**：一层 LoRA 有多少个参数？以 DistilBERT 前馈网络的第一层 `lin1`（768 → 3072）、秩 $r = 8$ 为例，把 `n_lora` 写成算式；再算出它占完整 $\Delta W$ 的百分之几。

In [ ]:
# 在这里写你的代码
raise NotImplementedError
print(f"LoRA r=8 的参数 {n_lora:,} | 完整 ΔW {768 * 3072:,} | 占 {100 * n_lora / (768 * 3072):.2f}%")
ck.lora_param_count(n_lora, 8, 768, 3072)

## E3 · 把你的 LoRALinear 装进 DistilBERT，训练一次

讲义里只验证了"装上 LoRA 前后输出不变"，还没训练过它。现在：

1. 写 `add_lora(model, r, alpha)`：冻结全部参数 → 把 6 个 Transformer 层里的 `attention.q_lin`、`attention.v_lin` 换成你的 `LoRALinear` → 重新打开分类头（`pre_classifier`、`classifier`）的梯度；
2. 用下面给好的数据和训练循环训练 3 个 epoch（r = 4、alpha = 8、AdamW lr = 1e-3），报告测试准确率。

数据用 CPU 小配置：训练集前 300 条、测试集前 500 条、最长 96 个 token。

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME, revision=REVISION)
imdb_train, imdb_test = data.load_imdb("train"), data.load_imdb("test")
N_TRAIN, N_TEST, MAX_LEN = 300, 500, 96


def encode(texts):
    enc = tok(list(texts), truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


ids_tr, mask_tr = encode(imdb_train["text"][:N_TRAIN])
ids_te, mask_te = encode(imdb_test["text"][:N_TEST])
y_tr = torch.tensor(imdb_train["label"][:N_TRAIN].to_numpy())
y_te = torch.tensor(imdb_test["label"][:N_TEST].to_numpy())


def train_one_epoch(model, loader, optimizer):
    model.train()
    for ids, mask, yb in loader:
        out = model(input_ids=ids.to(device), attention_mask=mask.to(device), labels=yb.to(device))
        optimizer.zero_grad()
        out.loss.backward()
        optimizer.step()


def evaluate(model, ids, mask, y, batch_size=64):
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(y), batch_size):
            logits = model(input_ids=ids[i:i + batch_size].to(device),
                           attention_mask=mask[i:i + batch_size].to(device)).logits
            correct += (logits.argmax(dim=1).cpu() == y[i:i + batch_size]).sum().item()
    return correct / len(y)


print("训练", tuple(ids_tr.shape), "| 测试", tuple(ids_te.shape))

In [ ]:
def add_lora(model, r, alpha):
    # 在这里写你的代码
    raise NotImplementedError
    return model


experiments.set_seed(0)
lora_model = add_lora(AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, revision=REVISION, num_labels=2),
                      r=4, alpha=8).to(device)
print(f"可训练参数 {count_trainable(lora_model):,}（应为 A/B 73,728 + 分类头 592,130 = 665,858）")
optimizer = torch.optim.AdamW([p for p in lora_model.parameters() if p.requires_grad], lr=1e-3)
loader = DataLoader(TensorDataset(ids_tr, mask_tr, y_tr), batch_size=16, shuffle=True)
t0 = time.time()
for epoch in range(3):
    train_one_epoch(lora_model, loader, optimizer)
scratch_acc = evaluate(lora_model, ids_te, mask_te, y_te)
print(f"你的 LoRA（r=4）3 个 epoch：测试准确率 {scratch_acc:.3f} | 用时 {time.time() - t0:.0f} 秒")
ck.scratch_lora_accuracy(scratch_acc)

**追问**（一两句话）：训练 3 个 epoch 之后，第 1 层 `q_lin` 的 `B` 还是 0 吗？`base.weight` 动过没有？写两行代码验证你的回答（提示：`lora_model.distilbert.transformer.layer[0].attention.q_lin`）。

In [ ]:
# 在这里写你的代码
raise NotImplementedError


## E4 · 两个一维高斯的解析 KL

实现讲义 S3 的闭式解（注意参数是**标准差** $\sigma$，不是方差）：

$$\mathrm{KL}\big(\mathcal{N}(\mu_1, \sigma_1^2) \,\|\, \mathcal{N}(\mu_2, \sigma_2^2)\big) = \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

写完先自己验证三件事：$\mathrm{KL}(p\|p) = 0$；两个方差相同、均值差 1 时 KL = 0.5；交换两个分布，结果变不变？

In [ ]:
def gaussian_kl(mu1, sigma1, mu2, sigma2):
    # 在这里写你的代码
    raise NotImplementedError


print("KL(p‖p) =", gaussian_kl(0, 1, 0, 1), "| 均值差 1:", gaussian_kl(1, 1, 0, 1),
      "| KL(N(0,2²)‖N(0,1)) =", round(gaussian_kl(0, 2, 0, 1), 4),
      "| KL(N(0,1)‖N(0,2²)) =", round(gaussian_kl(0, 1, 0, 2), 4))
ck.gaussian_kl(gaussian_kl)

## E5 · 蒙特卡洛估计 KL

KL 是一个期望：$\mathrm{KL}(p\|q) = \mathbb{E}_{x \sim p}[\log p(x) - \log q(x)]$。写 `mc_kl(mu1, sigma1, mu2, sigma2, n, rng)`：从 $p$ 抽 `n` 个样本，返回 $\log p(x) - \log q(x)$ 的样本平均。对 $p = \mathcal{N}(2, 1.5^2)$、$q = \mathcal{N}(0, 1)$ 用 10 万个样本估计，和 E4 的解析值比较。

高斯的对数密度已经给出。

In [ ]:
def log_gauss(x, mu, sigma):
    return -0.5 * np.log(2 * np.pi * sigma ** 2) - (x - mu) ** 2 / (2 * sigma ** 2)


def mc_kl(mu1, sigma1, mu2, sigma2, n, rng):
    # 在这里写你的代码
    raise NotImplementedError


exact = gaussian_kl(2.0, 1.5, 0.0, 1.0)
estimate = mc_kl(2.0, 1.5, 0.0, 1.0, 100_000, np.random.default_rng(0))
print(f"解析 {exact:.4f} | 蒙特卡洛 {estimate:.4f}")
ck.mc_kl(estimate, exact)

**E5b · 两个方向，哪个更难估？** 对 $p = \mathcal{N}(1, 0.5^2)$、$q = \mathcal{N}(0, 1)$，分别估计 $\mathrm{KL}(p\|q)$ 和 $\mathrm{KL}(q\|p)$：每个方向在 $n = 1000$ 个样本下重复估计 200 次，算出 200 个估计值的标准差 `spread_pq`、`spread_qp`，画两个直方图。哪个方向的蒙特卡洛估计更"抖"？试着用 $\log p - \log q$ 在采样分布下的形状解释（提示：$q$ 比 $p$ 宽，从 $q$ 抽出来的样本会落到 $p$ 的尾巴上）。

In [ ]:
gen = np.random.default_rng(1)
# 在这里写你的代码
raise NotImplementedError
print(f"KL(p‖q)：解析 {gaussian_kl(1.0, 0.5, 0.0, 1.0):.4f}，估计的标准差 {spread_pq:.4f}")
print(f"KL(q‖p)：解析 {gaussian_kl(0.0, 1.0, 1.0, 0.5):.4f}，估计的标准差 {spread_qp:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(est_pq, bins=30, alpha=0.6, label="KL(p || q), samples from p")
ax.hist(est_qp, bins=30, alpha=0.6, label="KL(q || p), samples from q")
ax.set_xlabel("Monte Carlo estimate (n = 1000)")
ax.set_ylabel("Count (200 repeats)")
ax.set_title("Sampling from the wider distribution gives a noisier KL estimate")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

（在这里写你的解释）